In [1]:
import os
# Only go up a directory if we are currently in the 'research' folder
if os.getcwd().endswith('research'):
  os.chdir("../")

In [2]:
%pwd

'/Users/manish/Desktop/text-summarizer'

In [3]:
from dataclasses import dataclass
from pathlib import Path

@dataclass(frozen=True)
class DataTransformationConfig:
  root_dir: Path
  data_path: Path
  tokenizer_name: Path

In [4]:
from textSummarizer.constants import *
from textSummarizer.utils.common import read_yaml, create_directories

In [5]:
class ConfigurationManager:
  def __init__(
    self,
    config_filepath = CONFIG_FILE_PATH,
    params_filepath = PARAMS_FILE_PATH):

    self.config = read_yaml(config_filepath)
    self.params = read_yaml(params_filepath)

    create_directories([self.config.artifacts_root])

  def get_data_transformation_config(self) -> DataTransformationConfig:
    config = self.config.data_transformation 

    create_directories([config.root_dir])

    data_transformation_config = DataTransformationConfig(
      root_dir=config.root_dir,
      data_path=config.data_path,
      tokenizer_name=config.tokenizer_name
    )

    return data_transformation_config



In [ ]:
import os 
from textSummarizer.logging import logging
from transformers import AutoTokenizer
from datasets import load_dataset, load_from_disk

class DataTransformation:
    def __init__(self, config: DataTransformationConfig):
        self.config = config
        self.tokenizer = AutoTokenizer.from_pretrained(config.tokenizer_name)

    def convert_examples_to_features(self, example_batch):
        input_encodings = self.tokenizer(example_batch['dialogue'], max_length=1024, truncation=True)
        
        target_encodings = self.tokenizer(text_target=example_batch['summary'], max_length=128, truncation=True)
            
        return {
            'input_ids': input_encodings['input_ids'],
            'attention_mask': input_encodings['attention_mask'],
            'labels': target_encodings['input_ids']
        }

    def convert(self):
        dataset_samsum = load_from_disk(self.config.data_path)
        dataset_samsum_pt = dataset_samsum.map(self.convert_examples_to_features, batched=True)
        dataset_samsum_pt.save_to_disk(os.path.join(self.config.root_dir, 'samsum_dataset'))


In [10]:
try:
  config = ConfigurationManager()
  data_transformation_config = config.get_data_transformation_config()
  data_transformation = DataTransformation(config=data_transformation_config)
  data_transformation.convert()
except Exception as e:
  raise e

[2026-10-12 17:45:13,752:INFO:common]: yaml file: config/config.yaml loaded successfully
[2026-10-12 17:45:13,753:INFO:common]: yaml file: params.yaml loaded successfully
[2026-10-12 17:45:13,754:INFO:common]: created directory at: artifacts
[2026-10-12 17:45:13,754:INFO:common]: created directory at: artifacts/data_transformation
[2026-10-12 17:45:14,150:INFO:_client]: HTTP Request: GET https://huggingface.co/api/models/google/pegasus-cnn_dailymail "HTTP/1.1 200 OK"
[2026-10-12 17:45:14,413:INFO:_client]: HTTP Request: GET https://huggingface.co/api/models/google/pegasus-cnn_dailymail/tree/main/additional_chat_templates?recursive=false&expand=false "HTTP/1.1 404 Not Found"
[2026-10-12 17:45:14,764:INFO:_client]: HTTP Request: GET https://huggingface.co/api/models/google/pegasus-cnn_dailymail/tree/main?recursive=true&expand=false "HTTP/1.1 200 OK"


Saving the dataset (1/1 shards): 100%|██████████| 818/818 [00:00<00:00, 442873.46 examples/s]
